# Data gathering and preprocessing

In [4]:
bash ./InstallLibraries.ps1

SyntaxError: invalid syntax (861591055.py, line 1)

In [10]:
import kagglehub
import pandas as pd

# Download latest version
kagglehub.dataset_download("jessemostipak/hotel-booking-demand", output_dir="./Dataset")

'./Dataset'

In [15]:
ds = pd.read_csv("./Dataset/hotel_bookings.csv")

In [18]:
print(ds.nunique(dropna=False))

# for col in ds.columns:
#     print(f"\nColumn: {col}")
#     print(ds[col].value_counts(dropna=False))

hotel                                2
is_canceled                          2
lead_time                          479
arrival_date_year                    3
arrival_date_month                  12
arrival_date_week_number            53
arrival_date_day_of_month           31
stays_in_weekend_nights             17
stays_in_week_nights                35
adults                              14
children                             6
babies                               5
meal                                 5
country                            178
market_segment                       8
distribution_channel                 5
is_repeated_guest                    2
previous_cancellations              15
previous_bookings_not_canceled      73
reserved_room_type                  10
assigned_room_type                  12
booking_changes                     21
deposit_type                         3
agent                              334
company                            353
days_in_waiting_list     

In [24]:
missingCountsTotal = ds.isna().sum()
missingCountsPercent = (missingCountsTotal / len(ds)) * 100

print("Missing values per column (count | percent):")
for col in ds.columns:
    count = missingCountsTotal[col]
    percent = missingCountsPercent[col]
    print(f"{col}: {count} | {percent:.2f}%")

Missing values per column (count | percent):
hotel: 0 | 0.00%
is_canceled: 0 | 0.00%
lead_time: 0 | 0.00%
arrival_date_year: 0 | 0.00%
arrival_date_month: 0 | 0.00%
arrival_date_week_number: 0 | 0.00%
arrival_date_day_of_month: 0 | 0.00%
stays_in_weekend_nights: 0 | 0.00%
stays_in_week_nights: 0 | 0.00%
adults: 0 | 0.00%
children: 4 | 0.00%
babies: 0 | 0.00%
meal: 0 | 0.00%
country: 488 | 0.41%
market_segment: 0 | 0.00%
distribution_channel: 0 | 0.00%
is_repeated_guest: 0 | 0.00%
previous_cancellations: 0 | 0.00%
previous_bookings_not_canceled: 0 | 0.00%
reserved_room_type: 0 | 0.00%
assigned_room_type: 0 | 0.00%
booking_changes: 0 | 0.00%
deposit_type: 0 | 0.00%
agent: 16340 | 13.69%
company: 112593 | 94.31%
days_in_waiting_list: 0 | 0.00%
customer_type: 0 | 0.00%
adr: 0 | 0.00%
required_car_parking_spaces: 0 | 0.00%
total_of_special_requests: 0 | 0.00%
reservation_status: 0 | 0.00%
reservation_status_date: 0 | 0.00%


In [25]:
ds.drop(columns=["company"], inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# 1) separate
train_df = ds[ds["agent"].notna()].copy()
predict_df = ds[ds["agent"].isna()].copy()

# 2) target and features
y = train_df["agent"].astype(str)  # if numeric keep as int
X = train_df.drop(columns=["agent"])
X_pred = predict_df.drop(columns=["agent"])

# 3) choose feature types
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["number"]).columns.tolist()

# 4) pipeline for encoding
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

model = Pipeline([
    ("pre", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])

agent_counts = y.value_counts()
good_agents = agent_counts[agent_counts >= 2].index
mask = train_df["agent"].isin(good_agents)
X = X[mask]; y = y[mask]

# 5) train/val split for quick eval
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model.fit(X_train, y_train)
print("val score:", model.score(X_val, y_val))

# 6) predict missing agent
predicted_agents = model.predict(X_pred)

# 7) fill
ds.loc[ds["agent"].isna(), "agent"] = predicted_agents
print("missing now:", ds["agent"].isna().sum())

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.